In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, confusion_matrix
from tqdm import tqdm
import time

In [2]:
data = pd.read_csv('../dataset/final_dataset_diff_f_L10.csv', parse_dates=['GAME_DATE'], dtype={'gameId' : str, 'H_teamId' : str, 'A_teamId' : str,})
data = data.round(2)

In [3]:
condition = (data['GAME_DATE'] > pd.to_datetime('2023-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2024-09-01'))
data_test = data[condition]
#data_test = data_test.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])
data_train = data[~condition]
#data_train = data_train.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])

In [4]:
X_train = data_train.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName','trueShootingPercentage_L10', 'PCT_TIR_REUSSI_L10', 'effectiveFieldGoalPercentage_L10','NB_WIN_L10', 'PCT_3PT_L10'])  # Fonctionnalités
y_train = data_train['HOME_WON']  # Cible

X_test = data_test.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName','trueShootingPercentage_L10', 'PCT_TIR_REUSSI_L10', 'effectiveFieldGoalPercentage_L10', 'NB_WIN_L10', 'PCT_3PT_L10'])  # Fonctionnalités
y_test = data_test['HOME_WON']

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
X_test

array([[0.41129139, 0.49425287, 0.33333333, ..., 0.76632302, 0.76647564,
        0.49295775],
       [0.4432422 , 0.54597701, 0.60606061, ..., 0.48911798, 0.48949379,
        0.57676056],
       [0.60875817, 0.77586207, 0.63636364, ..., 0.15521191, 0.15520535,
        0.49507042],
       ...,
       [0.51946317, 0.66091954, 0.75757576, ..., 0.16036655, 0.1599809 ,
        0.59366197],
       [0.5942564 , 0.62068966, 0.45454545, ..., 0.3722795 , 0.37249284,
        0.53943662],
       [0.49087775, 0.45402299, 0.39393939, ..., 0.5395189 , 0.53963706,
        0.56619718]])

In [5]:
#knn = KNeighborsClassifier(n_neighbors=41, weights='distance', metric='minkowski', p=2 )
#knn.fit(X_train, y_train)

# Initialiser le modèle k-NN
knn = KNeighborsClassifier()

# Définir la grille de paramètres à tester
param_grid = {
    'n_neighbors': [ 15, 16, 17],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1, 2], 
    'metric': ['euclidean', 'manhattan', 'minkowski'],
}

In [ ]:
# Calculer le nombre total d'itérations
total_iterations = len(param_grid['n_neighbors']) * len(param_grid['weights']) * len(param_grid['algorithm']) * len(param_grid['metric']) * len(param_grid['p'])
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
# Créer une barre de progression avec tqdm
with tqdm(total=total_iterations) as pbar:
    # Fonction de mise à jour de la barre de progression
    def update_pbar(*args):
        pbar.update()

    # Remplacer la méthode de mise à jour des barres de progression par notre fonction
    grid_search.fit = update_pbar.__get__(grid_search.fit, GridSearchCV)

    # Lancer la recherche d'hyperparamètres
    grid_search.fit(X_train, y_train)
# Utiliser GridSearchCV pour trouver les meilleures combinaisons de paramètres
# Attendre la fin de la barre de progression
while pbar.n != total_iterations:
    time.sleep(0.1)
#grid_search.fit(X_train, y_train)
print("a")

 50%|█████     | 1/2 [00:00<00:00, 12157.40it/s]


In [ ]:
# Afficher les meilleurs paramètres trouvés
print("Meilleurs paramètres trouvés : ", grid_search.best_params_)
print("Meilleure précision moyenne sur l'ensemble d'entraînement : ", grid_search.best_score_)

In [ ]:
#y_pred = knn.predict(X_test) 
# Évaluer le modèle sur le jeu de test
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)  # y_test sont les étiquettes de classe réelles des données de test
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1-score:", f1)
print("Matrice de confusion :")
print(conf_matrix)

In [ ]:
data_knn_predict = data_test[['gameId', 'GAME_DATE', 'HOME_WON', 'H_POINTS', 'A_POINTS','H_teamName', 'A_teamName']].copy()
data_knn_predict.loc[:, 'PRED'] = y_pred
#data_knn_predict = data_knn_predict.iloc[[2,3, 50,51, 207,208]]
data_knn_predict = data_knn_predict[['gameId', 'GAME_DATE', 'HOME_WON', 'PRED', 'H_POINTS', 'A_POINTS','H_teamName', 'A_teamName']]
data_knn_predict

In [ ]:
data_knn_predict.set_index('gameId', inplace=True)
data_knn_predict = data_knn_predict.sort_values(by='GAME_DATE')
data_knn_predict.transpose().to_json("../dataset/KNN_predict.json")